# 05 · Duplicates and clustering

- **Duplicates:** embed with `task="similarity"`, then `eg.near_duplicates` lists every pair above a threshold.
- **Clustering:** embed with `task="clustering"`, then use any clustering method on the vectors.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

In [2]:
import numpy as np

engine = eg.EmbeddingEngine()

## Finding near-duplicates
Two of these notes say the same thing in different words.

In [3]:
notes = [
    "Most plastic ever made has never been recycled; it sits in landfill or breaks into microplastics.",
    "Bloom the coffee for thirty seconds, then pour slowly in circles.",
    "The majority of all plastic produced was never recycled and ends up in dumps or as microplastics.",
    "Giraffes eat acacia leaves high above other animals.",
    "Zebras and giraffes often graze side by side.",
]
vectors = engine.embed(notes, task="similarity")

for pair in eg.near_duplicates(vectors, threshold=0.9):
    print(f"{pair.score:.3f}  #{pair.i} ~ #{pair.j}")

0.966  #0 ~ #2


### Picking the threshold
Unrelated texts still score fairly high (here 0.7–0.73), and related-but-different texts can come
close to duplicates: the giraffe and zebra notes (#3, #4) score 0.895. Look at the scores of *all*
pairs in your data and pick a threshold that separates the real duplicates.

In [4]:
all_pairs = eg.near_duplicates(vectors, threshold=-1)
for pair in all_pairs:
    print(f"{pair.score:.3f}  #{pair.i} ~ #{pair.j}")

0.966  #0 ~ #2
0.895  #3 ~ #4
0.732  #0 ~ #4
0.722  #1 ~ #4
0.721  #1 ~ #3
0.721  #0 ~ #3
0.720  #0 ~ #1
0.719  #2 ~ #4
0.719  #1 ~ #2
0.708  #2 ~ #3


## Grouping photos
A tiny k-means (any library works, e.g. scikit-learn) on `clustering` vectors.

In [5]:
photos = ["dishes.jpg", "kitchen.jpg", "groceries.jpg", "merchandise.jpg", "merchandise2.jpg",
          "wildlife.jpg", "aquatic.jpg", "human.jpg", "girl.jpg", "crowd.jpg", "trash.jpg"]
X = engine.embed([eg.Item.image(IMAGES / p) for p in photos], task="clustering").vectors


def kmeans(X, k, steps=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(steps):
        group = np.argmax(X @ centers.T, axis=1)
        centers = np.stack([X[group == c].mean(0) if (group == c).any() else centers[c] for c in range(k)])
        centers /= np.linalg.norm(centers, axis=1, keepdims=True)
    return group


groups = kmeans(X, k=4)
for g in range(4):
    print(f"group {g}:", ", ".join(p for p, x in zip(photos, groups) if x == g))

group 0: groceries.jpg, merchandise.jpg, merchandise2.jpg
group 1: human.jpg, girl.jpg, crowd.jpg, trash.jpg
group 2: dishes.jpg, kitchen.jpg, aquatic.jpg
group 3: wildlife.jpg


## Cluster one kind of media at a time
Photos, videos and texts all share one space, and a photo is closest to its own caption.
But in a mixed pile the **kind** of media is the strongest difference, so clustering
everything together mostly gives "the photos", "the videos" and "the texts".
Cluster each kind separately, and use search to connect them (notebook 03).

## Common mistakes

In [6]:
print("Comparing similarity vectors with clustering vectors:")
other = engine.embed(["x"], task="clustering")
mistake(lambda: eg.similarity(vectors, other))

print("A threshold outside -1…1:")
mistake(lambda: eg.near_duplicates(vectors, threshold=90))

Comparing similarity vectors with clustering vectors:
IncompatibleEmbeddingsError: Cannot compare these embeddings: task similarity vs clustering

A threshold outside -1…1:
InputError: threshold must be in [-1, 1], got 90.

